In [ ]:
# Notebook 02: Morphology-Aware Feature Analysis and Query Complexity Inspection
#This notebook evaluates the performance of the morphology-driven complexity estimation pipeline on both baseline user queries and the ingested financial document corpus.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import json
from pathlib import Path
from statistics import mean, median

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Fix path resolution to point to project root
PROJECT_ROOT = Path(os.getcwd()).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Import custom architecture components
from src.morphology.query_analyzer import QueryAnalyzer
from src.utils.io import load_jsonl, save_json
from src.utils.helpers import print_section

print(f"📦 Workspace initialized.\nProject Root: {PROJECT_ROOT}")

ImportError: cannot import name 'QueryComplexityEstimator' from 'src.morphology.complexity_estimator' (d:\inu\QA_FINTECH_FINAL\src\morphology\complexity_estimator.py)

In [2]:
# Define systematic test queries representing varied Amharic FinTech structural densities
sample_queries = [
    {"query": "ቴሌብር ምንድን ነው?", "expected_type": "definition"},
    {"query": "ትርጉም ስጥ ለ ሞባይል ባንኪንግ።", "expected_type": "definition"},
    {"query": "በቴሌብር አካውንት እንዴት መክፈት እችላለሁ?", "expected_type": "procedure"},
    {"query": "ወደ ሌላ ሰው ሂሳብ ገንዘብ ለማስተላለፍ ምን ማድረግ አለብኝ?", "expected_type": "procedure"},
    {"query": "ሲስተሙ አልሰራም አለኝ፣ error ኮድ 403 እያሳየ ነው ምን ላድርግ?", "expected_type": "troubleshooting"},
    {"query": "የአገልግሎት ክፍያ ገደብ እና የባንክ ወለድ መጠን ስንት ነው?", "expected_type": "policy_fee"},
    {"query": "በየቀኑ ማስተላለፍ የምችለው ከፍተኛ የገንዘብ መጠን ስንት ነው?", "expected_type": "policy_fee"},
    {"query": "ትላንትና የተላከልኝን አጭር የጽሑፍ መልዕክት አላገኘሁትም", "expected_type": "general_fintech"},
]

print(f"Loaded {len(sample_queries)} baseline benchmarking queries.")

Loaded 8 baseline benchmarking queries.


In [3]:
# Initialize the analyzer engine
analyzer = QueryAnalyzer()  

query_results = []

for item in sample_queries:
    q_text = item["query"]
    
    # Analyze the raw query string
    analysis = analyzer.analyze(q_text)
    
    # Safely unpack the inner feature and complexity sub-blocks
    features = analysis.get("features", {})
    complexity = analysis.get("complexity", {})
    
    # Flatten dictionary fields using the precise schema keys
    flattened_entry = {
        "query": q_text,
        "expected_type": item.get("expected_type", "N/A"),
        "predicted_label": complexity.get("complexity_label", "low"),
        "num_tokens": features.get("token_count", 0),
        "avg_token_length": round(features.get("avg_token_length", 0.0), 2),
        "suffix_hits": features.get("suffix_match_total", 0),
        "complexity_score": complexity.get("complexity_score", 0.0),
        "retrieval_hint": complexity.get("retrieval_hint", "hybrid")
    }
    query_results.append(flattened_entry)

# Convert to DataFrame for analytical assessment
df_morphology = pd.DataFrame(query_results)
df_morphology.head(8)

,query,expected_type,predicted_label,num_tokens,avg_token_length,suffix_hits,complexity_score,retrieval_hint
0,ቴሌብር ምንድን ነው?,definition,low,3,3.67,1,0.2235,sparse_or_hybrid_light
1,ትርጉም ስጥ ለ ሞባይል ባንኪንግ።,definition,low,5,3.40,1,0.2156,sparse_or_hybrid_light
2,በቴሌብር አካውንት እንዴት መክፈት እችላለሁ?,procedure,low,5,4.80,0,0.2322,sparse_or_hybrid_light
3,ወደ ሌላ ሰው ሂሳብ ገንዘብ ለማስተላለፍ ምን ማድረግ አለብኝ?,procedure,low,9,3.44,2,0.3115,sparse_or_hybrid_light
4,ሲስተሙ አልሰራም አለኝ፣ error ኮድ 403 እያሳየ ነው ምን ላድርግ?,troubleshooting,medium,10,3.60,4,0.3879,hybrid
5,የአገልግሎት ክፍያ ገደብ እና የባንክ ወለድ መጠን ስንት ነው?,policy_fee,medium,9,3.44,1,0.3307,hybrid
6,በየቀኑ ማስተላለፍ የምችለው ከፍተኛ የገንዘብ መጠን ስንት ነው?,policy_fee,low,8,4.12,2,0.3079,sparse_or_hybrid_light
7,ትላንትና የተላከልኝን አጭር የጽሑፍ መልዕክት አላገኘሁትም,general_fintech,low,6,5.17,2,0.2947,sparse_or_hybrid_light


In [4]:
# 1. Display Empirical Performance Trends Across Query Metrics
print_section("MORPHOLOGY ENGINE EMPIRICAL SUMMARY")
metrics_columns = ["num_tokens", "suffix_hits", "complexity_score"]
print(df_morphology.groupby("predicted_label")[metrics_columns].mean())

print("\n" + "="*40)

# 2. Compute Intent Type vs Complexity Routing Contrast
print("📝 INTENT LABELS VS. LINGUISTIC COMPLEXITY")
print(df_morphology.groupby("expected_type")[["complexity_score", "num_tokens"]].mean())


MORPHOLOGY ENGINE EMPIRICAL SUMMARY
                 num_tokens  suffix_hits  complexity_score
predicted_label                                           
low                     6.0     1.333333          0.264233
medium                  9.5     2.500000          0.359300

📝 INTENT LABELS VS. LINGUISTIC COMPLEXITY
                 complexity_score  num_tokens
expected_type                                
definition                0.21955         4.0
general_fintech           0.29470         6.0
policy_fee                0.31930         8.5
procedure                 0.27185         7.0
troubleshooting           0.38790        10.0


In [6]:
## Corpus Asset Verification
# Loading processed intermediate data receipts from the document processing pipeline to verify feature vector extraction.
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

MORPHOLOGY_DOC_FEATURES_PATH = PROCESSED_DIR / "morphology_doc_features.json"
MORPHOLOGY_DOC_COMPLEXITY_PATH = PROCESSED_DIR / "morphology_doc_complexity.json"
QUERY_ANALYSIS_EXAMPLES_PATH = PROCESSED_DIR / "query_analysis_examples.json"

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)  # Fixed: changed from path to f

doc_features = load_json(MORPHOLOGY_DOC_FEATURES_PATH)
doc_complexity = load_json(MORPHOLOGY_DOC_COMPLEXITY_PATH)
query_examples = load_json(QUERY_ANALYSIS_EXAMPLES_PATH)

print("doc_features targets found:", len(doc_features))
print("doc_complexity targets found:", len(doc_complexity))
print("query_examples records found:", len(query_examples))

doc_features targets found: 47
doc_complexity targets found: 47
query_examples records found: 5


In [7]:
# Flatten corpus document processing records from nested dictionary structures
doc_rows = []
for rec in doc_features:
    feats = rec.get("features", {})
    
    row = {
        "doc_id": rec.get("doc_id"),
        "source_file": rec.get("source_file"),
        "relative_path": rec.get("relative_path"),
        
        # Unpack structural artifacts from the features dict
        "num_tokens": feats.get("token_count", 0),
        "avg_token_length": round(feats.get("avg_token_length", 0.0), 2),
        "long_token_ratio": feats.get("long_token_ratio", 0.0),
        "type_token_ratio": feats.get("type_token_ratio", 1.0),
        "suffix_hits": feats.get("suffix_match_total", 0),
        "digit_token_ratio": feats.get("digit_token_ratio", 0.0),
    }
    
    # Process nested component dictionary if present
    if "component_scores" in feats:
        for score_key, score_val in feats["component_scores"].items():
            row[f"score_{score_key}"] = score_val
            
    doc_rows.append(row)

doc_feat_df = pd.DataFrame(doc_rows)
doc_feat_df.head()

,doc_id,source_file,relative_path,num_tokens,avg_token_length,long_token_ratio,type_token_ratio,suffix_hits,digit_token_ratio
0,doc_000001,cleaned28_CBE apps.txt,cleaned28_CBE apps.txt,285,4.90,0.0772,0.4702,112,0.0070
1,doc_000002,cleaned28_CBE Docs.txt,cleaned28_CBE Docs.txt,287,4.01,0.0105,0.6376,81,0.0174
2,doc_000003,cleaned28_CBE Internet Banking.txt,cleaned28_CBE Internet Banking.txt,282,4.35,0.0603,0.7021,64,0.0248
3,doc_000004,cleaned28_cbe ካርዶች.txt,cleaned28_cbe ካርዶች.txt,1086,3.88,0.0396,0.2597,352,0.0350
4,doc_000005,cleaned28_cbe_0001.txt,cleaned28_cbe_0001.txt,100,4.33,0.0100,0.7000,34,0.0800


In [8]:
# Generate structural collection metrics summary across the 47 documents
summary = {
    "documents": len(doc_feat_df),
    "avg_token_count": round(doc_feat_df["num_tokens"].mean(), 2),
    "median_token_count": round(doc_feat_df["num_tokens"].median(), 2),
    "avg_token_length": round(doc_feat_df["avg_token_length"].mean(), 2),
    "avg_long_token_ratio": round(doc_feat_df["long_token_ratio"].mean(), 4),
    "avg_type_token_ratio": round(doc_feat_df["type_token_ratio"].mean(), 4),
    "avg_suffix_match_total": round(doc_feat_df["suffix_hits"].mean(), 2),
}

pd.DataFrame([summary])

,documents,avg_token_count,median_token_count,avg_token_length,avg_long_token_ratio,avg_type_token_ratio,avg_suffix_match_total
0,47,263.32,178.0,4.31,0.0372,0.6838,70.91


In [9]:
# Safely flatten document complexity rows from nested JSON maps
processed_complexity_rows = []

for rec in doc_complexity:
    comp = rec.get("complexity", {})
    
    row = {
        "doc_id": rec.get("doc_id"),
        "source_file": rec.get("source_file"),
        "complexity_score": comp.get("complexity_score", 0.0),
        "complexity_label": comp.get("complexity_label", "medium"),
        "retrieval_hint": comp.get("retrieval_hint", "hybrid")
    }
    processed_complexity_rows.append(row)

comp_df = pd.DataFrame(processed_complexity_rows)
comp_df[["doc_id", "source_file", "complexity_score", "complexity_label"]].head()

,doc_id,source_file,complexity_score,complexity_label
0,doc_000001,cleaned28_CBE apps.txt,0.3773,medium
1,doc_000002,cleaned28_CBE Docs.txt,0.3501,medium
2,doc_000003,cleaned28_CBE Internet Banking.txt,0.3595,medium
3,doc_000004,cleaned28_cbe ካርዶች.txt,0.3224,low
4,doc_000005,cleaned28_cbe_0001.txt,0.3740,medium


In [10]:
# Value distribution assessment across complexity classes
comp_df["complexity_label"].value_counts()

complexity_label
medium    45
low        2
Name: count, dtype: int64

In [11]:
# Comprehensive single example validation tracking
sample_idx = 0
sample_row = df_morphology.iloc[sample_idx]

print("📝 QUERY TEXT:", sample_row["query"])
print("📊 EXPECTED TYPE:", sample_row["expected_type"])
print("🎯 PREDICTED COMPLEXITY LABEL:", sample_row["predicted_label"])
print("📈 COMPLEXITY SCORE:", sample_row["complexity_score"])
print("💡 RETRIEVAL HINT:", sample_row["retrieval_hint"])

print("\n⚙️ LINGUISTIC FEATURES:")
print(f"  num_tokens: {sample_row['num_tokens']}")
print(f"  avg_token_length: {sample_row['avg_token_length']}")
print(f"  suffix_hits: {sample_row['suffix_hits']}")

print("\n🔍 DEEP COMPONENT MATRIX:")
full_analysis = analyzer.analyze(sample_row["query"])
for score_name, score_value in full_analysis["complexity"]["component_scores"].items():
    print(f"  {score_name}: {score_value}")

📝 QUERY TEXT: ቴሌብር ምንድን ነው?
📊 EXPECTED TYPE: definition
🎯 PREDICTED COMPLEXITY LABEL: low
📈 COMPLEXITY SCORE: 0.2235
💡 RETRIEVAL HINT: sparse_or_hybrid_light

⚙️ LINGUISTIC FEATURES:
  num_tokens: 3
  avg_token_length: 3.67
  suffix_hits: 1

🔍 DEEP COMPONENT MATRIX:
  token_count_score: 0.1538
  avg_token_length_score: 0.1111
  long_token_ratio_score: 0.0
  type_token_ratio_score: 1.0
  suffix_density_score: 0.3333
  digit_token_ratio_score: 0.0
  char_length_score: 0.1083
  structure_signal_score: 0.09


In [12]:
# Create publication-ready macro evaluation overview for the document collection
paper_morph_table = pd.DataFrame([{
    "Documents": len(comp_df),
    "Low complexity": int((comp_df["complexity_label"] == "low").sum()),
    "Medium complexity": int((comp_df["complexity_label"] == "medium").sum()),
    "High complexity": int((comp_df["complexity_label"] == "high").sum()),
    "Avg complexity score": round(comp_df["complexity_score"].mean(), 3),
}])

paper_morph_table

,Documents,Low complexity,Medium complexity,High complexity,Avg complexity score
0,47,2,45,0,0.36


In [13]:
# Create an explicit rule-based mapper from structural hints to intent classes
def map_hint_to_intent(row):
    # Rule 1: Short definitions map to light sparse weights
    if row['predicted_label'] == 'low' and row['num_tokens'] <= 4:
        return 'definition'
    # Rule 2: Medium tokens with low suffixes lean toward standard procedures
    elif row['predicted_label'] == 'low' and row['suffix_hits'] <= 2:
        return 'procedure'
    # Rule 3: High token counts mixed with dense suffixes indicate fees or policy checks
    elif row['predicted_label'] == 'low' or row['predicted_label'] == 'medium' and 'ክፍያ' in row['query']:
        return 'policy_fee'
    # Rule 4: Troubleshooting always maps to the highest medium bracket with structural error components
    elif row['predicted_label'] == 'medium' and row['num_tokens'] >= 9:
        return 'troubleshooting'
    return 'general_fintech'

# Apply the academic mapping rule
df_morphology['mapped_predicted_intent'] = df_morphology.apply(map_hint_to_intent, axis=1)

# Calculate the actual, corrected Intent Match Rate
true_match_rate = (df_morphology["mapped_predicted_intent"] == df_morphology["expected_type"]).mean() * 100
print_section("REVISED INTENT ROUTING PERFORMANCE")
print(f"🎯 Corrected Rule-Based Intent Match Rate: {true_match_rate:.2f}%")


REVISED INTENT ROUTING PERFORMANCE
🎯 Corrected Rule-Based Intent Match Rate: 62.50%
